# Experiment 6: Bidirectional LSTM

## Overview
- Model: Bidirectional LSTM (Keras / TensorFlow)
- Approach: Train with Stratified 5-Fold CV, save per-fold models and metrics, produce averaged test predictions for submission.
- Artifacts (per model tracking guide): model files `model_fold_i.h5`, `tokenizer.json`, `metrics.json`, and submission CSV `exp_6_bilstm_submission.csv`.

In [ ]:
use_drive = False

# uncomment if running on colab
# from google.colab import drive
# drive.mount('/content/drive')
# use_drive = True

Mounted at /content/drive


In [2]:
# %pip install tensorflow

In [3]:
# BiLSTM imports and basic configuration
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, log_loss
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [4]:
# Initialize paths
DATA_PATH = 'data/'
EXPERIMENT_PATH = 'experiments/'
EXPERIMENT_NAME = 'exp_6_bilstm'

if use_drive:
    DRIVE_PATH = '/content/drive/MyDrive/Kaggle'
    EXPERIMENT_PATH = os.path.join(DRIVE_PATH, EXPERIMENT_PATH)

EXPERIMENT_DIR = os.path.join(EXPERIMENT_PATH, EXPERIMENT_NAME)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR, 'models'), exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)

Experiment directory: /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm


In [5]:
# Load data
train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

In [6]:
# Prepare word-level tokenizer for BiLSTM
# Use previously loaded `train` and `test` DataFrames
texts = train['text'].astype(str).values
test_texts = test['text'].astype(str).values
labels = train['label'].values

# Tokenizer at word level; lower-case to normalize
word_tokenizer = Tokenizer(lower=True, oov_token='<UNK>')
word_tokenizer.fit_on_texts(texts)
vocab_size = len(word_tokenizer.word_index)
print(f'Word vocab size: {vocab_size}')

# Save tokenizer mapping
tokenizer_path = os.path.join(EXPERIMENT_DIR, 'tokenizer.json')
with open(tokenizer_path, 'w') as f:
    json.dump({'word_index': word_tokenizer.word_index, 'word_level': True}, f)
print('Saved tokenizer to', tokenizer_path)

Word vocab size: 17518
Saved tokenizer to /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm/tokenizer.json


In [7]:
# Convert texts to sequences and pad
# Choose a reasonable maxlen for word sequences (shorter than char-level)
MAXLEN = 200  # adjust depending on dataset and memory
X_seq = word_tokenizer.texts_to_sequences(texts)
X_test_seq = word_tokenizer.texts_to_sequences(test_texts)
X_pad = pad_sequences(X_seq, maxlen=MAXLEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAXLEN, padding='post', truncating='post')
print('Train shape:', X_pad.shape)
print('Test shape:', X_test_pad.shape)

Train shape: (4712, 200)
Test shape: (1178, 200)


In [8]:
# Model builder: Bidirectional LSTM
def build_bilstm(vocab_size, maxlen, embed_dim=128, lstm_units=128):
    inp = layers.Input(shape=(maxlen,), dtype='int32')
    x = layers.Embedding(input_dim=vocab_size + 1, output_dim=embed_dim, mask_zero=True)(inp)
    x = layers.SpatialDropout1D(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(lstm_units // 2, dropout=0.2, recurrent_dropout=0.2))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Quick sanity check: build and show summary (comment out if running many folds)
m = build_bilstm(vocab_size, MAXLEN)
m.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 128)  │  2,242,432 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 200, 128)  │          0 │ embedding[0][0]   │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200, 256)  │    263,168 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │    164,352 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,678,273 (10.22 MB)

 Trainable params: 2,678,273 (10.22 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Cross-validation training loop
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

fold_metrics = []
fold_predictions = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_pad, labels), 1):
    print(f'\nFold {fold}/{n_folds}')
    X_tr, X_val = X_pad[train_idx], X_pad[val_idx]
    y_tr, y_val = labels[train_idx], labels[val_idx]

    model = build_bilstm(vocab_size, MAXLEN, embed_dim=128, lstm_units=128)

    model_path = os.path.join(EXPERIMENT_DIR, 'models', f'model_fold_{fold}.keras')
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        callbacks.ModelCheckpoint(model_path, monitor='val_loss', save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=64,
        callbacks=cb,
        verbose=1
    )

    # Validation predictions (use the model with restored best weights from EarlyStopping)
    y_val_proba = model.predict(X_val, batch_size=256).ravel()
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    metrics = {
        'fold': int(fold),
        'accuracy': float(accuracy_score(y_val, y_val_pred)),
        'precision': float(precision_score(y_val, y_val_pred, zero_division=0)),
        'recall': float(recall_score(y_val, y_val_pred, zero_division=0)),
        'f1': float(f1_score(y_val, y_val_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_val, y_val_proba)),
        'log_loss': float(log_loss(y_val, y_val_proba)),
        'train_size': int(len(train_idx)),
        'val_size': int(len(val_idx))
    }
    fold_metrics.append(metrics)
    print('Fold metrics:', metrics)

    # Test predictions (model already has best weights from EarlyStopping callback)
    test_proba = model.predict(X_test_pad, batch_size=256).ravel()
    fold_predictions += test_proba / n_folds

    # Save fold info
    print(f'Model for fold {fold} saved to {model_path}')

print('\nCross-validation complete')


Fold 1/5
Epoch 1/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.7238 - loss: 0.5646 - val_accuracy: 0.9385 - val_loss: 0.1685 - learning_rate: 0.0010
Epoch 2/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.9732 - loss: 0.0961 - val_accuracy: 0.9714 - val_loss: 0.0904 - learning_rate: 0.0010
Epoch 3/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - accuracy: 0.9964 - loss: 0.0176 - val_accuracy: 0.9671 - val_loss: 0.1124 - learning_rate: 0.0010
Epoch 4/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.9926 - loss: 0.0215 - val_accuracy: 0.9692 - val_loss: 0.1094 - learning_rate: 0.0010
Epoch 5/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - accuracy: 0.9995 - loss: 0.0069 - val_accuracy: 0.9629 - val_loss: 0.1531 - learning_rate: 5.0000e-04
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 796ms/step
Fold metrics: {'fold': 1, 'accuracy': 0.9713679745493107, 'precision': 0.9686411149825784, 'recall': 0.9391891891891891, 'f1': 0.9536878216123499, 'roc_auc': 0.9941518024980158, 'log_loss': 0.

In [10]:
# Summarize fold metrics and identify best fold
metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

avg_metrics = {}
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    avg_metrics[col] = float(metrics_df[col].mean())
    avg_metrics[f'{col}_std'] = float(metrics_df[col].std())

best_fold_idx = metrics_df['roc_auc'].idxmax()
best_fold = int(metrics_df.loc[best_fold_idx, 'fold'])
print(f'BEST FOLD: {best_fold} (ROC AUC: {metrics_df.loc[best_fold_idx, "roc_auc"]:.4f})')

# Save metrics.json according to tracking spec
metrics_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'BiLSTM (Keras)',
    'vectorizer': 'word_tokenizer',
    'vectorizer_params': {'word_level': True, 'lower': True, 'oov_token': '<UNK>'},
    'model_params': {'maxlen': MAXLEN, 'embed_dim': 128, 'lstm_units': 128},
    'n_folds': n_folds,
    'best_fold': best_fold,
    'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'average_metrics': avg_metrics,
    'fold_metrics': fold_metrics,
}
with open(os.path.join(EXPERIMENT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print('Saved metrics to', os.path.join(EXPERIMENT_DIR, 'metrics.json'))

 fold  accuracy  precision   recall       f1  roc_auc  log_loss  train_size  val_size
    1  0.971368   0.968641 0.939189 0.953688 0.994152  0.090406        3769       943
    2  0.980912   0.969595 0.969595 0.969595 0.998512  0.047275        3769       943
    3  0.968153   0.952218 0.945763 0.948980 0.991256  0.105068        3770       942
    4  0.968153   0.949324 0.949324 0.949324 0.991727  0.099526        3770       942
    5  0.960722   0.930233 0.945946 0.938023 0.990791  0.108368        3770       942
BEST FOLD: 2 (ROC AUC: 0.9985)
Saved metrics to /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm/metrics.json


In [11]:
# Create submission using averaged predictions
submission = pd.DataFrame({'id': test['id'], 'probability': fold_predictions})
submission_path = os.path.join(EXPERIMENT_DIR, f'{EXPERIMENT_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)
print('Saved submission to', submission_path)
print('Submission head:')
print(submission.head())

Saved submission to /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm/exp_6_bilstm_submission.csv
Submission head:
   id  probability
0   0     0.000668
1   1     0.000503
2   2     0.000399
3   3     0.001847
4   4     0.005916


In [12]:
# print model statistics
print('\nPrediction statistics:')
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    print(f' - {col}: {avg_metrics[col]:.4f} ± {avg_metrics[f"{col}_std"]:.4f}')



Prediction statistics:
 - accuracy: 0.9699 ± 0.0073
 - precision: 0.9540 ± 0.0162
 - recall: 0.9500 ± 0.0116
 - f1: 0.9519 ± 0.0114
 - roc_auc: 0.9933 ± 0.0032
 - log_loss: 0.0901 ± 0.0249


## Notes and Next Steps
- The BiLSTM uses word-level tokenization instead of character-level, which is more appropriate for capturing semantic relationships.
- The model uses bidirectional LSTMs to capture context from both directions, with two stacked layers for hierarchical feature learning.
- Includes SpatialDropout1D and recurrent dropout for regularization, plus ReduceLROnPlateau callback for better convergence.
- Tune `MAXLEN`, `embed_dim`, and `lstm_units` to balance performance and memory.
- Consider adding attention mechanisms or pretrained embeddings (Word2Vec, GloVe) for improved performance.
- Add callbacks for TensorBoard or Optuna-based hyperparameter tuning if needed.
- Verify `experiments/exp_6_bilstm/` contains: `models/model_fold_i.h5`, `tokenizer.json`, `metrics.json`, and submission CSV.